# Lab 1: 제조 지식 문서를 ChromaDB에 인덱싱

이 실습에서는 제조 현장 문서를 벡터 데이터베이스(ChromaDB)에 저장하여
의미 기반 검색(Semantic Search)이 가능하도록 인덱싱합니다.

## 학습 목표

1. **RAG 시스템에서 ChromaDB의 역할** 이해
2. ChromaDB 클라이언트 생성 및 컬렉션 관리
3. Sentence Transformer를 사용한 **한국어 임베딩** 생성
4. 제조 도메인 문서의 벡터 저장 및 유사도 검색
5. 검색 결과 시각화 및 인덱스 **영구 저장/로드**

## RAG 파이프라인에서 이 Lab의 위치

```
[이 Lab]
문서 텍스트 → 임베딩 모델 → 벡터 → ChromaDB 저장
                                        ↓
[다음 Lab]                       질문 → 유사도 검색 → 관련 문서 → LLM → 답변
```

> **비유**: ChromaDB는 도서관의 색인 카드 시스템입니다.  
> 책 내용을 숫자 벡터로 변환해 저장하고, 질문과 가장 비슷한 내용을 빠르게 찾아줍니다.

## 1. 환경 설정

In [ ]:
# 필요한 패키지 설치 (최초 1회만 실행)
# !pip install chromadb sentence-transformers pandas

In [ ]:
# 라이브러리 임포트
import chromadb
from sentence_transformers import SentenceTransformer
import json
import os
import pandas as pd

print("✅ 라이브러리 임포트 완료")
print(f"ChromaDB 버전: {chromadb.__version__}")

## 2. 제조 도메인 샘플 문서 생성

실제 현장에서는 설비 매뉴얼 PDF, 품질 기준서, 작업 표준서 등을 사용합니다.  
이 Lab에서는 제조 현장의 실제 지식을 담은 샘플 문서 12개를 사용합니다.

In [ ]:
# 제조 도메인 샘플 문서 12개 정의
# 실제 현장에서는 PDF, Word 등 파일에서 로드합니다
manufacturing_docs = [
    {
        "id": "doc01",
        "text": "CNC 밀링머신 진동 이상 징후: 스핀들 베어링 마모 시 100~300Hz 대역에서 비정상적인 피크가 발생합니다. 진동값이 3mm/s를 초과하면 즉시 가동을 중단하고 점검이 필요합니다. SKF-6205 베어링 기준.",
        "source": "설비매뉴얼",
        "topic": "진동이상",
        "equipment": "CNC밀링"
    },
    {
        "id": "doc02",
        "text": "용접 품질 검사 기준: 기공(Porosity) 결함은 X-ray 검사에서 원형 음영으로 나타나며, KS B 0845 기준으로 직경 3mm 이상의 기공은 불합격 처리합니다. 초음파 검사 주기는 월 1회 이상 실시합니다.",
        "source": "품질기준서",
        "topic": "용접품질",
        "equipment": "용접기"
    },
    {
        "id": "doc03",
        "text": "AutoEncoder 기반 이상탐지 원리: 정상 운전 데이터만으로 학습하여 재구성 오차(Reconstruction Error)가 임계값을 초과하면 이상으로 판정합니다. 임계값은 학습 데이터 표준편차의 3배로 설정합니다.",
        "source": "AI기술가이드",
        "topic": "이상탐지",
        "equipment": "전체"
    },
    {
        "id": "doc04",
        "text": "스마트공장 도입 ROI 사례(KAMP): 중소 제조업체 300개사 평균 생산성 33.6% 향상, 품질 불량률 44.4% 감소. 초기 구축 비용 1~3억원, 평균 투자 회수 기간 2.8년으로 분석됩니다.",
        "source": "KAMP사례집",
        "topic": "스마트공장ROI",
        "equipment": "전체"
    },
    {
        "id": "doc05",
        "text": "프레스 금형 마모 점검 기준: 타발 횟수 50만회 도달 시 금형 마모 점검을 실시합니다. 버(Burr) 높이가 0.3mm 초과하거나 치수 편차가 ±0.1mm를 넘으면 금형 교체가 필요합니다.",
        "source": "작업표준서",
        "topic": "금형관리",
        "equipment": "프레스"
    },
    {
        "id": "doc06",
        "text": "컨베이어 벨트 장력 관리: 처짐량이 벨트 길이의 1~3% 범위를 유지해야 합니다. 장력이 부족하면 슬립이 발생하고, 과도하면 베어링 수명이 단축됩니다. 분기별 1회 장력 측정 및 조정.",
        "source": "설비매뉴얼",
        "topic": "컨베이어관리",
        "equipment": "컨베이어"
    },
    {
        "id": "doc07",
        "text": "표면 결함 AI 검사 (YOLOv8): 스크래치, 크랙, 이물질 3가지 결함 유형을 실시간으로 탐지합니다. 신뢰도(Confidence) 0.85 이상만 불량으로 판정하여 오탐을 최소화합니다.",
        "source": "AI기술가이드",
        "topic": "비전검사",
        "equipment": "검사장비"
    },
    {
        "id": "doc08",
        "text": "오일 점도 관리: 윤활유 교체 주기는 운전 시간 기준 2000시간 또는 6개월 중 먼저 도달하는 시점에 실시합니다. 점도 지수(VI)가 80 미만으로 저하되면 즉시 교체해야 합니다.",
        "source": "설비매뉴얼",
        "topic": "윤활관리",
        "equipment": "전체"
    },
    {
        "id": "doc09",
        "text": "예지보전(PdM) 구현 단계: 1단계 데이터 수집(IoT 센서), 2단계 이상탐지 모델(AutoEncoder/FFT), 3단계 수명예측(RUL 모델), 4단계 정비 자동 스케줄링. 단계별 ROI 측정이 중요합니다.",
        "source": "AI기술가이드",
        "topic": "예지보전",
        "equipment": "전체"
    },
    {
        "id": "doc10",
        "text": "사출성형기 온도 관리: 배럴 온도는 설정값 대비 ±5°C 이내를 유지해야 합니다. 온도 편차가 10°C 이상 발생하면 히터 또는 쿨링 시스템 점검이 필요합니다. 온도 로그는 1분 단위 기록.",
        "source": "설비매뉴얼",
        "topic": "온도관리",
        "equipment": "사출성형기"
    },
    {
        "id": "doc11",
        "text": "전기 모터 효율 등급: IE3(프리미엄 효율) 이상 사용을 권장합니다. 전류 불균형이 5% 초과 시 권선 절연 열화를 의심하고 절연저항 측정(목표: 1MΩ 이상)을 실시합니다.",
        "source": "전기설비기준",
        "topic": "모터관리",
        "equipment": "전기모터"
    },
    {
        "id": "doc12",
        "text": "FFT 주파수 분석으로 베어링 결함 진단: BPFO(외륜결함) = 0.4 × 회전수 × 볼수, BPFI(내륜결함) = 0.6 × 회전수 × 볼수. 해당 주파수에서 스펙트럼 피크가 나타나면 결함으로 판정합니다.",
        "source": "진단기술서",
        "topic": "FFT진단",
        "equipment": "회전기계"
    }
]

print(f"✅ 샘플 문서 {len(manufacturing_docs)}개 준비 완료")
print("\n문서 목록:")
for doc in manufacturing_docs:
    print(f"  [{doc['id']}] {doc['topic']} | {doc['source']} | {doc['equipment']}")

## 3. 임베딩 모델 로드

**`paraphrase-multilingual-MiniLM-L12-v2`** 모델을 사용합니다.
- 한국어 포함 50개 언어 지원
- 빠른 속도 (384차원 벡터)
- 오프라인 사용 가능 (최초 다운로드 후 캐시)

> 더 높은 품질이 필요하면: `paraphrase-multilingual-mpnet-base-v2` (768차원, 더 느림)

In [ ]:
# 한국어 지원 다국어 임베딩 모델 로드
# 최초 실행 시 모델 다운로드 (~100MB, 약 1~2분 소요)
print("임베딩 모델 로드 중... (최초 실행 시 시간이 걸립니다)")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 모델 정보 출력
test_embedding = model.encode(["테스트"])[0]
print(f"✅ 모델 로드 완료")
print(f"  - 임베딩 차원: {len(test_embedding)}")
print(f"  - 지원 언어: 50개 언어 (한국어 포함)")

## 4. ChromaDB 클라이언트 선택

ChromaDB는 두 가지 모드를 제공합니다:

| 모드 | API | 특징 | 사용 케이스 |
|------|-----|------|-------------|
| 메모리 | `EphemeralClient()` | 재시작 시 초기화 | 학습, 테스트 |
| 영구 저장 | `PersistentClient(path=...)` | 디스크 저장, 재시작 후 복원 | 실서비스 배포 |

In [ ]:
# === ChromaDB 클라이언트 생성 ===

# [옵션 A] 메모리 전용 클라이언트 (실습용)
# 노트북 재시작 시 데이터가 초기화됩니다
client = chromadb.EphemeralClient()
db_mode = "메모리(EphemeralClient)"

# [옵션 B] 영구 저장 클라이언트 (서비스 배포용)
# 아래 주석을 해제하면 ./chroma_db 폴더에 영구 저장됩니다
# os.makedirs("./chroma_db", exist_ok=True)
# client = chromadb.PersistentClient(path="./chroma_db")
# db_mode = "영구저장(PersistentClient)"

print(f"✅ ChromaDB 클라이언트 생성: {db_mode}")
print(f"  chromadb 버전: {chromadb.__version__}")

## 5. 컬렉션 생성 및 문서 추가

In [ ]:
# 컬렉션 생성 (기존 컬렉션이 있으면 삭제 후 재생성)
COLLECTION_NAME = "manufacturing_docs_v1"

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"기존 컬렉션 '{COLLECTION_NAME}' 삭제")
except Exception:
    pass  # 컬렉션이 없으면 그냥 넘어감

# 새 컬렉션 생성
# cosine 유사도 사용 (텍스트 검색에 적합)
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}  # 코사인 유사도 사용
)
print(f"✅ 컬렉션 '{COLLECTION_NAME}' 생성 완료")

In [ ]:
# 임베딩 생성 (모든 문서를 한 번에 처리 - 배치 처리)
print("임베딩 생성 중...")
texts = [doc["text"] for doc in manufacturing_docs]
embeddings = model.encode(texts, show_progress_bar=True).tolist()

print(f"\n✅ 임베딩 생성 완료")
print(f"  - 문서 수: {len(texts)}개")
print(f"  - 임베딩 차원: {len(embeddings[0])}")
print(f"  - 첫 번째 임베딩 샘플: {embeddings[0][:5]}...")

In [ ]:
# ChromaDB에 문서, 임베딩, 메타데이터 저장
collection.add(
    ids=[doc["id"] for doc in manufacturing_docs],        # 고유 ID
    embeddings=embeddings,                                  # 벡터 (우리가 생성)
    documents=[doc["text"] for doc in manufacturing_docs], # 원본 텍스트
    metadatas=[                                             # 메타데이터 (출처 정보)
        {
            "source": doc["source"],
            "topic": doc["topic"],
            "equipment": doc["equipment"]
        }
        for doc in manufacturing_docs
    ]
)

print(f"✅ ChromaDB에 {collection.count()}개 문서 저장 완료")
print(f"   컬렉션 이름: {COLLECTION_NAME}")

## 6. 유사도 검색 실습

쿼리(질문) 텍스트를 임베딩으로 변환한 후, ChromaDB에서 가장 유사한 문서를 찾습니다.

In [ ]:
def search_docs(query: str, n_results: int = 3, filter_source: str = None) -> pd.DataFrame:
    """
    쿼리 텍스트로 유사한 문서를 검색합니다.
    
    Args:
        query: 검색할 질문/키워드
        n_results: 반환할 결과 수 (기본 3개)
        filter_source: 특정 출처만 검색 (예: '설비매뉴얼')
    
    Returns:
        검색 결과 DataFrame
    """
    # 쿼리 임베딩 생성
    query_embedding = model.encode([query]).tolist()
    
    # 필터 조건 설정 (선택사항)
    where_filter = {"source": filter_source} if filter_source else None
    
    # ChromaDB 검색
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )
    
    # 결과를 DataFrame으로 변환
    rows = []
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        # cosine 거리 → 유사도 변환 (1 - 거리)
        similarity = 1 - dist
        rows.append({
            "순위": i + 1,
            "유사도": f"{similarity:.3f}",
            "주제": meta['topic'],
            "출처": meta['source'],
            "설비": meta['equipment'],
            "내용(앞 80자)": doc[:80] + "..."
        })
    
    return pd.DataFrame(rows)

print("✅ 검색 함수 정의 완료")

In [ ]:
# 검색 테스트 1: 베어링 이상 진동 관련 쿼리
query1 = "베어링 진동 이상 기준이 뭐야?"
print(f"🔍 쿼리: '{query1}'")
print("-" * 70)

result1 = search_docs(query1, n_results=3)
print(result1.to_string(index=False))

In [ ]:
# 검색 테스트 2: AI 이상탐지 기술 관련 쿼리
query2 = "머신러닝으로 설비 이상을 어떻게 탐지하나요?"
print(f"🔍 쿼리: '{query2}'")
print("-" * 70)

result2 = search_docs(query2, n_results=3)
print(result2.to_string(index=False))

In [ ]:
# 검색 테스트 3: 스마트공장 투자 효과 쿼리
query3 = "스마트공장 도입하면 비용 대비 효과가 있나요?"
print(f"🔍 쿼리: '{query3}'")
print("-" * 70)

result3 = search_docs(query3, n_results=3)
print(result3.to_string(index=False))

In [ ]:
# 검색 테스트 4: 특정 출처 필터링 (설비매뉴얼만)
query4 = "정기 점검 주기와 교체 기준"
print(f"🔍 쿼리: '{query4}' (필터: 설비매뉴얼만)")
print("-" * 70)

result4 = search_docs(query4, n_results=3, filter_source="설비매뉴얼")
print(result4.to_string(index=False))

## 7. 검색 결과 시각화

다양한 쿼리에 대한 Top-K 검색 결과를 비교 테이블로 확인합니다.

In [ ]:
# 여러 쿼리를 한눈에 비교하는 시각화
test_queries = [
    "CNC 머신 베어링 교체 기준",
    "용접 불량 검사 방법",
    "예지보전 구현 단계",
    "오일 교체 주기",
    "FFT 주파수로 결함 진단",
]

print("=" * 70)
print("다중 쿼리 검색 결과 (Top-1 결과만 표시)")
print("=" * 70)

summary_rows = []
for query in test_queries:
    # 각 쿼리에 대해 Top-1 결과 검색
    q_emb = model.encode([query]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=1,
                           include=["documents", "metadatas", "distances"])
    
    doc = res['documents'][0][0]
    meta = res['metadatas'][0][0]
    similarity = 1 - res['distances'][0][0]
    
    summary_rows.append({
        "쿼리": query,
        "매칭 주제": meta['topic'],
        "출처": meta['source'],
        "유사도": f"{similarity:.3f}",
        "내용 요약": doc[:60] + "..."
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## 8. 인덱스 영구 저장 및 로드

실제 서비스에서는 PersistentClient로 인덱스를 디스크에 저장해두고,
서버가 재시작되어도 데이터가 유지되도록 합니다.

In [ ]:
# 영구 저장 클라이언트로 인덱스 저장
PERSIST_PATH = "./chroma_db_manufacturing"
os.makedirs(PERSIST_PATH, exist_ok=True)

# PersistentClient로 동일한 데이터 저장
persistent_client = chromadb.PersistentClient(path=PERSIST_PATH)

# 기존 컬렉션이 있으면 삭제
try:
    persistent_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

# 컬렉션 생성 및 데이터 저장
persistent_collection = persistent_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
persistent_collection.add(
    ids=[doc["id"] for doc in manufacturing_docs],
    embeddings=embeddings,
    documents=[doc["text"] for doc in manufacturing_docs],
    metadatas=[
        {"source": doc["source"], "topic": doc["topic"], "equipment": doc["equipment"]}
        for doc in manufacturing_docs
    ]
)

print(f"✅ 영구 저장 완료: {PERSIST_PATH}")
print(f"   저장된 문서 수: {persistent_collection.count()}개")

# 저장된 파일 목록 확인
print(f"\n저장된 파일들:")
for f in os.listdir(PERSIST_PATH):
    print(f"  - {f}")

In [ ]:
# 저장된 인덱스 로드 및 검색 확인
# (서버 재시작 후 이 코드만 실행해도 데이터가 복원됩니다)
print("저장된 인덱스 로드 중...")
loaded_client = chromadb.PersistentClient(path=PERSIST_PATH)
loaded_collection = loaded_client.get_collection(COLLECTION_NAME)

print(f"✅ 인덱스 로드 완료: {loaded_collection.count()}개 문서")

# 로드된 인덱스로 검색 테스트
test_query = "전동기 절연 불량 점검 방법"
test_emb = model.encode([test_query]).tolist()
test_result = loaded_collection.query(
    query_embeddings=test_emb, n_results=2,
    include=["documents", "metadatas", "distances"]
)

print(f"\n로드된 인덱스 검색 테스트: '{test_query}'")
for i, (doc, meta, dist) in enumerate(zip(
    test_result['documents'][0],
    test_result['metadatas'][0],
    test_result['distances'][0]
)):
    print(f"  [{i+1}] 유사도={1-dist:.3f} | {meta['topic']} | {doc[:60]}...")

## 9. ChromaDB 주요 기능 정리

In [ ]:
# ChromaDB 유용한 기능들 정리

# (1) 전체 문서 조회
print("[1] 전체 문서 수:", collection.count())

# (2) 특정 ID로 문서 조회
doc_by_id = collection.get(ids=["doc01"], include=["documents", "metadatas"])
print(f"\n[2] doc01 조회: {doc_by_id['documents'][0][:50]}...")

# (3) 문서 업데이트
collection.update(
    ids=["doc01"],
    documents=["[업데이트] CNC 밀링머신 진동 기준: SKF-6205 베어링 기준 진동값 3mm/s 초과 시 즉시 가동 중단."],
    embeddings=model.encode(["[업데이트] CNC 밀링머신 진동 기준"]).tolist()
)
print("\n[3] doc01 업데이트 완료")

# (4) 컬렉션 목록 확인
collections = client.list_collections()
print(f"\n[4] 현재 컬렉션 목록: {[c.name for c in collections]}")

print("\n✅ ChromaDB 기능 시연 완료")

## 10. 정리 및 다음 단계

### 이 Lab에서 배운 것

**핵심 흐름**:
```
문서 텍스트  →  SentenceTransformer  →  벡터(384차원)  →  ChromaDB 저장
질문 텍스트  →  SentenceTransformer  →  벡터(384차원)  →  코사인 유사도 검색  →  관련 문서 Top-K
```

**ChromaDB 클라이언트 선택 가이드**:

| 상황 | 추천 클라이언트 |
|------|----------------|
| 학습·테스트 | `EphemeralClient()` |
| 단일 서버 서비스 | `PersistentClient(path=...)` |
| 분산 서비스 | `HttpClient(host=..., port=...)` |

**다음 Lab (02_rag_pipeline.ipynb)**:
- 이 ChromaDB 인덱스를 활용하여 RAG 전체 파이프라인을 완성합니다
- 검색된 문서 → LLM → 답변 생성까지 연결합니다
- `ManufacturingRAG` 클래스를 구현하고 제조 AI 질문으로 테스트합니다